In [2]:
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType
from qdrant_client.models import Filter, FieldCondition, MatchValue

BACKEND_DIR = Path("..").resolve()
KB_JSON = BACKEND_DIR / "knowledge_base" / "knowledge_base.json"
UPITI_JSON = BACKEND_DIR / "knowledge_base" / "tekst_upiti.json"
SREDNJI_VEKTORI_NPZ = BACKEND_DIR / "knowledge_base" / "bge_srednji_vektori.npz"

QDRANT_URL = "http://localhost:6333"
COLLECTION = "plant_text"

MODEL_NAME = "BAAI/bge-large-en-v1.5"
VECTOR_SIZE = 1024
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PREFIKS_UPITA = "Represent this sentence for searching relevant passages: "

MAX_TOKENA = 500
TEZINA_DOKUMENTA = 0.3

print("DEVICE =", DEVICE)

DEVICE = cuda


In [3]:
with open(KB_JSON, encoding="utf-8") as f:
    knowledge_base = json.load(f)

print("Broj vrsta:", len(knowledge_base))
print(knowledge_base["0000"]["scientific_name"])

Broj vrsta: 324
Lactuca virosa


In [4]:

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
tokenizer = model.tokenizer

def broj_tokena(tekst):
    return len(tokenizer.encode(tekst, add_special_tokens=False))

In [5]:
def ocisti_naslove(tekst):
    tekst = re.sub(r"\s*^=+[^\n]*=+\s*\Z", "", tekst, flags=re.MULTILINE)
    tekst = re.sub(r"^=+\s*(.+?)\s*=+[ \t]*\n\s*", r"\1: ", tekst, flags=re.MULTILINE)
    return tekst.strip()

In [13]:
KRAJ_RECENICE = r"(?<=[.!?])\s+(?=[A-Z])"

SEPARATORI = ["\n\n\n", "\n\n", "\n", KRAJ_RECENICE]


def podeli(tekst, separator):
    if separator == KRAJ_RECENICE:
        delovi = re.split(KRAJ_RECENICE, tekst)
    else:
        delovi = tekst.split(separator)
    rezultat = []
    for deo in delovi:
        if deo.strip() != "":
            rezultat.append(deo.strip())
    return rezultat 

In [ ]:

def podeli_na_chunkove(tekst, nivo=0):
    if broj_tokena(tekst) <= MAX_TOKENA:
        return [tekst]

    separator = SEPARATORI[nivo]
    celine = podeli(tekst, separator)

    if len(celine) == 1:
        return podeli_na_chunkove(tekst, nivo + 1)

    delovi = []
    for celina in celine:
        if broj_tokena(celina) <= MAX_TOKENA:
            delovi.append(celina)
        else:
            delovi += podeli_na_chunkove(celina, nivo + 1)

    if separator == KRAJ_RECENICE:
        spojnica = " "
    else:
        spojnica = separator
    return spoji_u_chunkove(delovi, spojnica)

    
def spoji_u_chunkove(celine, spojnica):
    ukupno = 0
    for celina in celine:
        ukupno += broj_tokena(celina)
    broj_delova = math.ceil(ukupno / MAX_TOKENA)
    cilj = ukupno / broj_delova

    chunkovi = []
    tekuci = []
    tekuci_tokeni = 0
    for celina in celine:
        n = broj_tokena(celina)
        if len(tekuci) > 0:
            novo = tekuci_tokeni + n
            if novo > MAX_TOKENA or abs(novo - cilj) > abs(tekuci_tokeni - cilj):
                chunkovi.append(spojnica.join(tekuci))
                tekuci = []
                tekuci_tokeni = 0
        tekuci.append(celina)
        tekuci_tokeni += n
    chunkovi.append(spojnica.join(tekuci))

    if len(chunkovi) > 1 and broj_tokena(chunkovi[-2]) + broj_tokena(chunkovi[-1]) <= MAX_TOKENA:
        poslednji = chunkovi.pop()
        chunkovi[-1] = chunkovi[-1] + spojnica + poslednji
    return chunkovi

In [ ]:

def recenice(tekst):
    lista = []
    for red in podeli(tekst, "\n"):
        lista += podeli(red, KRAJ_RECENICE)
    return lista

def prozori(tekst):
    r = recenice(tekst)
    if len(r) <= 2:
        return [" ".join(r)]
    lista = []
    for i in range(len(r) - 1):
        lista.append(r[i] + " " + r[i + 1])
    return lista

In [8]:
records = []
for specie_id in sorted(knowledge_base):
    zapis = knowledge_base[specie_id]
    tekst = ocisti_naslove(zapis["tekst"])

    for nivo in ["dokument", "prozor"]:
        if nivo == "dokument":
            chunkovi = podeli_na_chunkove(tekst)
        else:
            chunkovi = prozori(tekst)

        for i, chunk in enumerate(chunkovi):
            records.append({
                "id": len(records),
                "payload": {
                    "specie_id": specie_id,
                    "scientific_name": zapis["scientific_name"],
                    "nivo": nivo,
                    "chunk_index": i,
                    "chunk_num": len(chunkovi),
                    "text": chunk,
                    "broj_tokena": broj_tokena(chunk),
                    "izvor": zapis["izvor"],
                },
            })

df = pd.DataFrame([r["payload"] for r in records])

print("Ukupno tačaka:", len(df))
print(df["nivo"].value_counts())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1108 > 512). Running this sequence through the model will result in indexing errors


Ukupno tačaka: 2858
nivo
prozor      2509
dokument     349
Name: count, dtype: int64


In [10]:
def centriraj(v, srednji_vektor):
    v = v - srednji_vektor
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


tekstovi = [r["payload"]["text"] for r in records]
nivoi = np.array([r["payload"]["nivo"] for r in records])

sirovi_embeddings = model.encode(tekstovi, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True)

with open(UPITI_JSON, encoding="utf-8") as f:
    svi_upiti = json.load(f)

val_upiti = []
for specie_id in svi_upiti:
    if specie_id == "_meta":
        continue
    for stavka in svi_upiti[specie_id]["val"]:
        val_upiti.append(PREFIKS_UPITA + stavka["upit"])

srednji_vektor_prozora = sirovi_embeddings[nivoi == "prozor"].mean(axis=0)
srednji_vektor_dokumenata = sirovi_embeddings[nivoi == "dokument"].mean(axis=0)
srednji_vektor_val_upita = model.encode(val_upiti, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True).mean(axis=0)

centrirani_embeddings = sirovi_embeddings.copy()
centrirani_embeddings[nivoi == "prozor"] = centriraj(sirovi_embeddings[nivoi == "prozor"], srednji_vektor_prozora)
centrirani_embeddings[nivoi == "dokument"] = centriraj(sirovi_embeddings[nivoi == "dokument"], srednji_vektor_dokumenata)

np.savez(SREDNJI_VEKTORI_NPZ, srednji_vektor_prozora=srednji_vektor_prozora, srednji_vektor_dokumenata=srednji_vektor_dokumenata, srednji_vektor_val_upita=srednji_vektor_val_upita)
print("Sačuvano:", SREDNJI_VEKTORI_NPZ.name)

Batches:   0%|          | 0/179 [00:00<?, ?it/s]

Batches: 100%|██████████| 41/41 [00:18<00:00,  2.23it/s]


Sačuvano: bge_srednji_vektori.npz


In [11]:
client = QdrantClient(url=QDRANT_URL)

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "centriran": VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
        "sirov": VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    },
)

client.create_payload_index(COLLECTION, field_name="specie_id", field_schema=PayloadSchemaType.KEYWORD)
client.create_payload_index(COLLECTION, field_name="nivo", field_schema=PayloadSchemaType.KEYWORD)

for start in range(0, len(records), BATCH_SIZE):
    points = []
    for i in range(start, min(start + BATCH_SIZE, len(records))):
        points.append(PointStruct(
            id=records[i]["id"],
            vector={"centriran": centrirani_embeddings[i].tolist(), "sirov": sirovi_embeddings[i].tolist()},
            payload=records[i]["payload"],
        ))
    client.upsert(collection_name=COLLECTION, points=points)

print("Tačaka u kolekciji:", client.count(COLLECTION).count)

Tačaka u kolekciji: 2858


In [12]:
def skor_po_vrsti(vektor, nivo):
    odgovor = client.query_points_groups(
        collection_name=COLLECTION,
        query=vektor.tolist(),
        using="centriran",
        query_filter=Filter(must=[FieldCondition(key="nivo", match=MatchValue(value=nivo))]),
        group_by="specie_id",
        limit=len(knowledge_base),
        group_size=1,
    )
    skorovi = {}
    for grupa in odgovor.groups:
        skorovi[grupa.id] = grupa.hits[0].score
    return skorovi


upit = ("yellow flowers with five petals dotted with black glands, "
        "opposite sessile leaves with translucent dots")
vektor = model.encode(PREFIKS_UPITA + upit, normalize_embeddings=True)
vektor = centriraj(vektor, srednji_vektor_val_upita)

prozor = skor_po_vrsti(vektor, "prozor")
dokument = skor_po_vrsti(vektor, "dokument")

skor = {}
for sid in prozor:
    skor[sid] = (1 - TEZINA_DOKUMENTA) * prozor[sid] + TEZINA_DOKUMENTA * dokument[sid]

najbolje = sorted(skor, key=skor.get, reverse=True)[:5]
for mesto, sid in enumerate(najbolje, start=1):
    print(mesto, knowledge_base[sid]["scientific_name"], round(skor[sid], 3))

1 Hypericum humifusum 0.437
2 Hypericum maculatum 0.406
3 Hypericum pulchrum 0.392
4 Hypericum hirsutum 0.383
5 Hypericum androsaemum 0.312
